# CCEA - All Samples

Generalises `CCE_1 _sample.ipynb` to run the Cooperative Coevolutionary Algorithm over **every**
malware sample in the dataset instead of just the first one.

- Input: `../data/CCE_AllSamples_long.csv` (17 samples, `CCE.csv` schema, distinguished by the `Hash` column).
- `Reduced_Severity` is still taken straight from the CSV (placeholder). The LLM hook is marked but not wired.
- The evaluation-metrics section at the end is a stub for later.

### Step 1 - Imports

In [ ]:
import math
import time
import random

from dataclasses import dataclass, field

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

### Step 2 - CCEA and score-mapping parameters

In [ ]:
# ----------------------------------------------------------
# CCEA PARAMETERS
# ----------------------------------------------------------
RANDOM_SEED           = 42
MAX_GENERATIONS       = 100
MUTATION_PROBABILITY  = 0.20
EXPLORATION_PROBABILITY = 0.25
PATIENCE              = 20
MIN_IMPROVEMENT       = 1e-8

# ----------------------------------------------------------
# SCORE MAPPING PARAMETERS  (applied only during fitness evaluation)
# ----------------------------------------------------------
MIN_MAPPED_SCORE       = 0.03   # epsilon  (Parameters!B2)
MAX_ATTACK_PLAN_SCORE  = 35     #          (Parameters!B3)
MAX_REDUCED_SEVERITY   = 10

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print('CCEA PARAMETERS')
print('-' * 50)
print('Random Seed             :', RANDOM_SEED)
print('Maximum Generations     :', MAX_GENERATIONS)
print('Mutation Probability    :', MUTATION_PROBABILITY)
print('Exploration Probability :', EXPLORATION_PROBABILITY)
print('Patience                :', PATIENCE)

### Step 3 - Load the multi-sample dataset

In [ ]:
# ---- Reduced_Severity source ----------------------------------------------
# False -> the Reduced_Severity column already in the CSV
# True  -> the LLM-produced variant (from AG Cuckoo Very new/evaluate.py)
USE_LLM = False
DATA_PATH = ('../data/CCE_AllSamples_long_llm.csv' if USE_LLM
             else '../data/CCE_AllSamples_long.csv')

data = pd.read_csv(DATA_PATH)

# Optional friendly labels for plots / tables (safe if a hash is missing).
SAMPLE_LABELS = {
    '15232': 'Sample01', '18327': 'Sample02', '71452': 'Sample03', '83925': 'Sample04',
    '19564': 'Sample05', '26137': 'Sample06', '29178': 'Sample07', '31247': 'Sample08',
    '90009': 'Sample09', '48791': 'Sample10', '58934': 'Sample11', '98265': 'Sample12',
    '98250': 'Sample13', '98248': 'Sample14', '98360': 'Sample15', '98241': 'Sample16',
    '98242': 'Sample17',
}

print('Dataset Shape :', data.shape)
print('Columns       :', data.columns.tolist())
print('Samples       :', data['Hash'].nunique())
display(data.head(10))

### Step 4 - Clean, verify and validate the whole dataset

In [ ]:
# A. clean column names
data.columns = data.columns.str.replace('\u00a0', ' ', regex=False).str.strip()

# B. required columns
required_columns = ['Candidate_ID', 'Hash', 'Signature_ID',
                    'Defense_ID', 'Attack_Plan_Score', 'Reduced_Severity']
missing = [c for c in required_columns if c not in data.columns]
if missing:
    raise ValueError(f'Missing required columns: {missing}')

# C. identifier columns as clean strings
for column in ['Candidate_ID', 'Hash', 'Signature_ID', 'Defense_ID']:
    data[column] = (data[column].astype(str).str.strip()
                    .str.replace(r'\.0$', '', regex=True))

# D. numeric columns
for column in ['Attack_Plan_Score', 'Reduced_Severity']:
    data[column] = pd.to_numeric(data[column], errors='coerce')

# E. per-sample unique Candidate_ID
dups = data.duplicated(subset=['Hash', 'Candidate_ID']).sum()
if dups:
    raise ValueError(f'{dups} rows have a duplicate (Hash, Candidate_ID).')

# F. no missing values
if data[['Defense_ID', 'Attack_Plan_Score', 'Reduced_Severity']].isnull().any().any() \
        or (data['Defense_ID'] == '').any():
    raise ValueError('Some rows have missing Defense_ID / Attack_Plan_Score / Reduced_Severity.')

# G-H. range checks
assert data['Attack_Plan_Score'].between(1, MAX_ATTACK_PLAN_SCORE).all(), 'Attack_Plan_Score out of range'
assert data['Reduced_Severity'].between(0, MAX_REDUCED_SEVERITY).all(), 'Reduced_Severity out of range'

print('DATASET CLEANING AND VALIDATION COMPLETED')
print('-' * 55)
print('Total Candidate Rows :', len(data))
print('Malware Samples       :', data['Hash'].nunique())
print('Unique Signatures     :', data['Signature_ID'].nunique())
print()
summary_counts = (data.groupby('Hash')
                  .agg(Signatures=('Signature_ID', 'nunique'),
                       Candidates=('Candidate_ID', 'count'))
                  .reset_index())
summary_counts['Sample'] = summary_counts['Hash'].map(SAMPLE_LABELS)
display(summary_counts[['Sample', 'Hash', 'Signatures', 'Candidates']])

### Step 5 - Defense candidate

In [ ]:
@dataclass
class DefenseCandidate:
    candidate_id: str
    signature_id: str
    defense_id: str
    attack_plan_score: float
    reduced_severity: float          # from the CSV - placeholder for the LLM output

    fitness_history: list = field(default_factory=list)
    evaluation_count: int = 0

    @property
    def mean_fitness(self):
        return float(np.mean(self.fitness_history)) if self.fitness_history else float('-inf')

    @property
    def best_fitness(self):
        return float(np.max(self.fitness_history)) if self.fitness_history else float('-inf')

### Step 6 - Subpopulations, representative selection, mutation, combination

In [ ]:
def create_subpopulations(sample_data, signature_order):
    """One defense subpopulation per triggered signature. Every CSV row is kept."""
    subpopulations = {}
    for signature_id in signature_order:
        rows = sample_data[sample_data['Signature_ID'] == signature_id]
        candidates = [
            DefenseCandidate(
                candidate_id=row['Candidate_ID'],
                signature_id=row['Signature_ID'],
                defense_id=str(row['Defense_ID']),
                attack_plan_score=float(row['Attack_Plan_Score']),
                reduced_severity=float(row['Reduced_Severity']),
            )
            for _, row in rows.iterrows()
        ]
        subpopulations[signature_id] = candidates
    return subpopulations


def calculate_rms(values):
    values = np.asarray(values, dtype=float)
    if values.size == 0:
        raise ValueError('Cannot calculate RMS from an empty list.')
    return float(np.sqrt(np.mean(np.square(values))))


def select_representative(candidates):
    unexplored = [c for c in candidates if c.evaluation_count == 0]
    if unexplored:
        return random.choice(unexplored), 'Unexplored'
    if random.random() < EXPLORATION_PROBABILITY:
        return random.choice(candidates), 'Random Exploration'
    best = max(candidates, key=lambda c: (c.mean_fitness, c.best_fitness))
    return best, 'Collaborative Fitness'


def mutate_candidate(selected, candidates):
    if len(candidates) <= 1:
        return selected, False
    if random.random() >= MUTATION_PROBABILITY:
        return selected, False
    alternatives = [c for c in candidates if c.candidate_id != selected.candidate_id]
    return random.choice(alternatives), True


def construct_combination(subpopulations):
    """Pick exactly one representative per signature -> one complete defense strategy."""
    combination = {}
    selection_log = []
    for signature_id, candidates in subpopulations.items():
        selected, method = select_representative(candidates)
        final, mutated = mutate_candidate(selected, candidates)
        combination[signature_id] = final
        selection_log.append({
            'Signature_ID': signature_id,
            'Initially_Selected': selected.candidate_id,
            'Final_Candidate': final.candidate_id,
            'Selection_Method': method,
            'Mutation': mutated,
        })
    return combination, selection_log

### Step 7 - Fitness evaluation (all mapping happens here)

```
AttackMapped   = eps + (1-eps) * (AttackRaw  - 1) / (35 - 1)
RSMapped       = ReducedSeverityRaw / 10
IS_Aggr        = RMS(AttackMapped)
RS_Aggr        = RMS(RSMapped)
Fitness        = (IS_Aggr - RS_Aggr) / IS_Aggr
```

`Reduced_Severity` comes from `candidate.reduced_severity` (the CSV placeholder). The single point
where an LLM would supply it instead is marked `# LLM HOOK` below - left untouched for now.

In [ ]:
def evaluate_combination(sample_hash, generation, combination):
    mapped_attack_scores = []
    mapped_reduced_severities = []
    combination_details = []

    for signature_id, candidate in combination.items():
        attack_mapped = (MIN_MAPPED_SCORE + (1 - MIN_MAPPED_SCORE)
                         * (candidate.attack_plan_score - 1) / (MAX_ATTACK_PLAN_SCORE - 1))

        # LLM HOOK: an LLM would return reduced severity here given
        #   (attack_mapped, complete attack plan, complete defense combination).
        # Until then we use the placeholder value carried on the candidate.
        reduced_severity_raw = candidate.reduced_severity
        reduced_severity_mapped = reduced_severity_raw / MAX_REDUCED_SEVERITY

        if not (0 <= attack_mapped <= 1):
            raise ValueError(f'Invalid mapped Attack Score for {candidate.candidate_id}')
        if not (0 <= reduced_severity_mapped <= 1):
            raise ValueError(f'Invalid mapped Reduced Severity for {candidate.candidate_id}')

        mapped_attack_scores.append(attack_mapped)
        mapped_reduced_severities.append(reduced_severity_mapped)
        combination_details.append({
            'Hash': sample_hash, 'Generation': generation,
            'Candidate_ID': candidate.candidate_id, 'Signature_ID': signature_id,
            'Defense_ID': candidate.defense_id,
            'Attack_Plan_Score': candidate.attack_plan_score, 'Attack_Mapped': attack_mapped,
            'Reduced_Severity': reduced_severity_raw,
            'Reduced_Severity_Mapped': reduced_severity_mapped,
        })

    IS_Aggr = calculate_rms(mapped_attack_scores)
    RS_Aggr = calculate_rms(mapped_reduced_severities)
    fitness = 0.0 if math.isclose(IS_Aggr, 0.0, abs_tol=1e-12) else (IS_Aggr - RS_Aggr) / IS_Aggr

    for candidate in combination.values():
        candidate.fitness_history.append(fitness)
        candidate.evaluation_count += 1

    return {
        'Hash': sample_hash, 'Generation': generation, 'Combination': combination,
        'Combination_Details': combination_details,
        'IS_Aggr': float(IS_Aggr), 'RS_Aggr': float(RS_Aggr), 'Fitness': float(fitness),
    }

### Step 8 - Run the CCEA for one sample

In [ ]:
def all_candidates_evaluated(subpopulations):
    return all(c.evaluation_count > 0
               for cs in subpopulations.values() for c in cs)


def run_ccea_for_sample(sample_hash, sample_data, seed=RANDOM_SEED, verbose=False):
    signature_order = list(dict.fromkeys(sample_data['Signature_ID'].tolist()))
    subpopulations = create_subpopulations(sample_data, signature_order)

    # fresh histories + deterministic per-sample stream
    for cs in subpopulations.values():
        for c in cs:
            c.fitness_history = []
            c.evaluation_count = 0
    random.seed(seed)
    np.random.seed(seed)

    search_space = 1
    for cs in subpopulations.values():
        search_space *= len(cs)

    best_result = None
    no_improve = 0
    generation_history = []
    generation = 0

    for generation in range(1, MAX_GENERATIONS + 1):
        combination, _ = construct_combination(subpopulations)
        result = evaluate_combination(sample_hash, generation, combination)

        improved = (best_result is None
                    or result['Fitness'] > best_result['Fitness'] + MIN_IMPROVEMENT)
        if improved:
            best_result = result
            no_improve = 0
        else:
            no_improve += 1

        generation_history.append({
            'Generation': generation,
            'Generation_Fitness': result['Fitness'],
            'Best_Fitness_So_Far': best_result['Fitness'],
            'IS_Aggr': result['IS_Aggr'],
            'RS_Aggr': result['RS_Aggr'],
        })

        if all_candidates_evaluated(subpopulations) and no_improve >= PATIENCE:
            break

    if verbose:
        print(f'{sample_hash}: gens={generation:3d}  best_fitness={best_result["Fitness"]:.6f}')

    return {
        'Hash': sample_hash,
        'n_signatures': len(signature_order),
        'n_candidates': int(sum(len(cs) for cs in subpopulations.values())),
        'search_space': search_space,
        'generations_run': generation,
        'best_result': best_result,
        'generation_history': pd.DataFrame(generation_history),
        'subpopulations': subpopulations,
    }

### Step 9 - Run every sample

In [ ]:
results_by_sample = {}

start = time.time()
for i, sample_hash in enumerate(data['Hash'].drop_duplicates().tolist()):
    sample_data = data[data['Hash'] == sample_hash].copy()
    res = run_ccea_for_sample(sample_hash, sample_data,
                              seed=RANDOM_SEED + i, verbose=True)
    results_by_sample[sample_hash] = res

print()
print(f'Completed {len(results_by_sample)} samples in {time.time() - start:.2f}s')

### Step 10 - Per-sample summary and across-sample statistics

In [ ]:
rows = []
for h, r in results_by_sample.items():
    b = r['best_result']
    rows.append({
        'Sample': SAMPLE_LABELS.get(h, ''),
        'Hash': h,
        'Signatures': r['n_signatures'],
        'Candidates': r['n_candidates'],
        'Search_Space': r['search_space'],
        'Generations_Run': r['generations_run'],
        'Best_Generation': b['Generation'],
        'IS_Aggr': round(b['IS_Aggr'], 6),
        'RS_Aggr': round(b['RS_Aggr'], 6),
        'Best_Fitness': round(b['Fitness'], 6),
    })

summary_df = pd.DataFrame(rows)
display(summary_df)

print('ACROSS-SAMPLE BEST FITNESS')
print('-' * 40)
print(summary_df['Best_Fitness'].describe()[['mean', 'std', 'min', '50%', 'max']]
      .rename({'50%': 'median'}).round(6).to_string())

### Step 11 - Convergence plots (one per sample)

In [ ]:
n = len(results_by_sample)
ncols = 4
nrows = math.ceil(n / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3 * nrows), squeeze=False)

for ax, (h, r) in zip(axes.flat, results_by_sample.items()):
    gh = r['generation_history']
    ax.plot(gh['Generation'], gh['Best_Fitness_So_Far'], marker='.', linewidth=1)
    ax.set_title(f"{SAMPLE_LABELS.get(h, h)}  (best={r['best_result']['Fitness']:.3f})", fontsize=9)
    ax.set_xlabel('Generation'); ax.set_ylabel('Best fitness')
    ax.grid(True, alpha=0.3)

for ax in axes.flat[n:]:
    ax.axis('off')

fig.suptitle('CCEA fitness convergence - all samples', y=1.02)
fig.tight_layout()
plt.show()

### Step 12 - Inspect one sample's optimal defense combination

In [ ]:
INSPECT_HASH = data['Hash'].iloc[0]   # change to any Hash value

best = results_by_sample[INSPECT_HASH]['best_result']
print('Sample        :', SAMPLE_LABELS.get(INSPECT_HASH, ''), INSPECT_HASH)
print('Best Generation:', best['Generation'])
print('IS_Aggr        :', round(best['IS_Aggr'], 6))
print('RS_Aggr        :', round(best['RS_Aggr'], 6))
print('Best Fitness   :', round(best['Fitness'], 6))
display(pd.DataFrame(best['Combination_Details']))

## Step 13 - Evaluation metrics (PENDING)

Not defined yet - mam will specify which metrics to report. Skeletons below so the wiring is ready:

- **Random-search baseline** - same number of full-combination evaluations, combinations drawn at random.
- **Greedy baseline** - per signature, pick the candidate with the lowest `Reduced_Severity`.
- **Multi-seed** - repeat each sample with several seeds, report mean +/- std of best fitness.

None of these are wired into a report yet.

In [ ]:
def greedy_combination(subpopulations):
    """Lower bound helper: pick min-reduced-severity candidate per signature."""
    return {sig: min(cs, key=lambda c: c.reduced_severity)
            for sig, cs in subpopulations.items()}


def random_search_best(subpopulations, n_evals, seed=0):
    rng = random.Random(seed)
    best = None
    for g in range(1, n_evals + 1):
        combo = {sig: rng.choice(cs) for sig, cs in subpopulations.items()}
        res = evaluate_combination('random', g, combo)
        if best is None or res['Fitness'] > best['Fitness']:
            best = res
    return best


def multi_seed_run(sample_hash, sample_data, seeds=range(10)):
    return [run_ccea_for_sample(sample_hash, sample_data, seed=s)['best_result']['Fitness']
            for s in seeds]


# TODO (mam): decide the metric set, then build the comparison table / plots here.
print('Evaluation section is a stub - see markdown above.')